# 03 – Feature Engineering & Machine Learning Modeling

**Part A (Supervised Learning):** Compare 4 classification algorithms (Logistic Regression, Decision Tree, Random Forest, Gradient Boosting) using 5-fold Stratified Cross-Validation and tune the winner with `RandomizedSearchCV`.  
**Part B (Unsupervised Learning):** Group customers using K-Means clustering and evaluate $K=2\dots8$ via the Elbow Method and Silhouette Score.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.pipeline import Pipeline

from ml import clustering, config, train
from ml.features import add_features, build_preprocessor

## 1. In-Pipeline Feature Engineering (`ml/features.py`)

In [ ]:
df = train.load_dataset()
X, y, input_cols = train.prepare_xy(df)
engineered = add_features(X)
new_cols = [c for c in engineered.columns if c not in X.columns]
print("Raw input columns:", input_cols)
print("Engineered columns added inside pipeline:", new_cols)
engineered[new_cols].describe().T.round(3)

## 2. Baseline & 5-Fold Cross-Validation Comparison (4 Classifiers)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE, stratify=y
)
dummy = Pipeline([("preprocess", build_preprocessor(X_train)), ("model", DummyClassifier(strategy="stratified", random_state=0))])
base_res = cross_validate(dummy, X_train, y_train, cv=config.CV_FOLDS, scoring=train.scoring())
print("Dummy Baseline CV:", {m: round(float(base_res[f'test_{m}'].mean()), 4) for m in train.scoring()})

cv_table = train.compare_models(X_train, y_train)
cv_table

## 3. Unsupervised K-Means Customer Segmentation ($K = 2\dots8$)

In [ ]:
seg_report = clustering.run(n_clusters=config.DEFAULT_K_CLUSTERS)
pd.DataFrame(seg_report["k_evaluation"])

In [ ]:
pd.DataFrame(seg_report["profiles"])[["cluster_id", "segment_name", "size", "share_pct", "conversion_rate_pct", "description"]]